In [1]:
from pyspark.sql import SparkSession
 
# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Processed NYC Taxis") \
    .getOrCreate()

In [27]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, year, month, round, unix_timestamp, sin, cos, log, exp, when, abs
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
import math

print("Iniciando PROCESSED: Entrenamiento con ventana de 3 meses y Validación...")

# LECTURA DE DATOS
ruta_cuarentena_ok = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/cuarentena/datos_con_columnas/"
df_crudo = spark.read.parquet(ruta_cuarentena_ok)

# LIMPIEZA Y KPIs
df_limpio = df_crudo.dropDuplicates().dropna(subset=["PULocationID", "DOLocationID"])
df_limpio = df_limpio.withColumn("duration_min", (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60)
df_limpio = df_limpio.filter((col("duration_min") >= 0.5) & (col("duration_min") <= 300) & (col("trip_distance") > 0) & (col("total_amount") >= 2.50))

df_kpis = df_limpio.groupBy(year("pickup_datetime").alias("Anio"), month("pickup_datetime").alias("Mes")).agg(count("*").alias("Total_Viajes"))

# PREPARACIÓN DEL MODELO (Índice Temporal)
df_ml = df_kpis.withColumn("Indice_Temporal", (col("Anio") - 2021) * 12 + col("Mes")).orderBy("Indice_Temporal")

# FILTRADO: ENTRENAMIENTO SOLO CON LOS ÚLTIMOS 3 MESES (Oct, Nov, Dic 2025)
# Suponiendo que el forecast es para Enero/Feb/Mar 2026, entrenamos con el final de 2025
print("Filtrando ventana de entrenamiento: Últimos 3 meses de 2025...")
df_entrenamiento_3m = df_ml.filter((col("Anio") == 2025) & (col("Mes") >= 10))

# Log Transform y Features
df_entrenamiento_3m = df_entrenamiento_3m.withColumn("label_log", log(col("Total_Viajes") + 1))
ensamblador = VectorAssembler(inputCols=["Indice_Temporal"], outputCol="features")

df_train_final = ensamblador.transform(df_entrenamiento_3m)

# Entrenamiento del modelo
modelo = LinearRegression(featuresCol="features", labelCol="label_log").fit(df_train_final)

# FORECAST: ENERO, FEBRERO Y MARZO 2026
# Creamos los índices para Ene(61), Feb(62), Mar(63) basándonos en el inicio 2021
futuro_datos = [(61, 2026, 1, "Enero 2026"), (62, 2026, 2, "Febrero 2026"), (63, 2026, 3, "Marzo 2026")]
df_futuro = spark.createDataFrame(futuro_datos, ["Indice_Temporal", "Anio", "Mes", "Periodo"])

# Predicción
df_prediccion = modelo.transform(ensamblador.transform(df_futuro))
df_forecast = df_prediccion.withColumn("Demanda_Predicha", round(exp(col("prediction")) - 1, 0).cast("integer")) \
                           .select("Anio", "Mes", "Periodo", "Demanda_Predicha")

print("\nPREDICCIÓN (FORECAST):")
df_forecast.show()

# COMPARACIÓN CON DATOS ORIGINALES (Enero y Febrero 2026)
print("Comparando con datos reales de Enero y Febrero...")

# Extraemos los reales del df_ml original
df_reales = df_ml.filter((col("Anio") == 2026) & (col("Mes").isin(1, 2, 3))) \
                 .select(col("Anio"), col("Mes"), col("Total_Viajes").alias("Demanda_Real"))

# Cruzamos Predicción vs Realidad
df_comparativa = df_forecast.join(df_reales, ["Anio", "Mes"], "inner") \
    .withColumn("Desviacion_Abs", abs(col("Demanda_Real") - col("Demanda_Predicha"))) \
    .withColumn("Error_Pct", round((col("Desviacion_Abs") / col("Demanda_Real")) * 100, 2))

print("\nRESULTADO DE LA COMPARACIÓN (REAL vs PREDICHO):")
df_comparativa.select("Periodo", "Demanda_Real", "Demanda_Predicha", "Error_Pct").show()

# GUARDADO
ruta_processed_forecast = "abfss://nyctaxi-processed@stnycanalysis.dfs.core.windows.net/processed/df_forecast_2026/"
ruta_comparativa = "abfss://nyctaxi-processed@stnycanalysis.dfs.core.windows.net/processed/df_comparativa_real_2026/"

df_forecast.write.mode("overwrite").parquet(ruta_processed_forecast)
df_comparativa.write.mode("overwrite").parquet(ruta_comparativa)

print("Pipeline finalizado con éxito.")

In [41]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, year, month, round, unix_timestamp, sin, cos, log, exp, when, abs
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
import math

print("Iniciando PROCESSED: Entrenamiento con ventana de 3 meses y Validación...")

# LECTURA DE DATOS
ruta_cuarentena_ok = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/cuarentena/datos_con_columnas/"
df_crudo = spark.read.parquet(ruta_cuarentena_ok)

# LIMPIEZA Y KPIs
df_limpio = df_crudo.dropDuplicates().dropna(subset=["PULocationID", "DOLocationID"])
df_limpio = df_limpio.withColumn("duration_min", (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60)
df_limpio = df_limpio.filter((col("duration_min") >= 0.5) & (col("duration_min") <= 300) & (col("trip_distance") > 0) & (col("total_amount") >= 2.50))

df_limpio.show(5)

In [28]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, year, month, round, unix_timestamp, sin, cos, log, exp, when, abs
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
import math

print("Iniciando PROCESSED: Entrenamiento con ventana de 3 meses y Validación...")

# LECTURA DE DATOS
ruta_cuarentena_ok = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/cuarentena/datos_con_columnas/"
df_crudo = spark.read.parquet(ruta_cuarentena_ok)

# LIMPIEZA Y KPIs
df_limpio = df_crudo.dropDuplicates().dropna(subset=["PULocationID", "DOLocationID"])
df_limpio = df_limpio.withColumn("duration_min", (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60)
df_limpio = df_limpio.filter((col("duration_min") >= 0.5) & (col("duration_min") <= 300) & (col("trip_distance") > 0) & (col("total_amount") >= 2.50))

df_kpis = df_limpio.groupBy(year("pickup_datetime").alias("Anio"), month("pickup_datetime").alias("Mes")).agg(count("*").alias("Total_Viajes"))

# PREPARACIÓN DEL MODELO (Índice Temporal)
df_ml = df_kpis.withColumn("Indice_Temporal", (col("Anio") - 2021) * 12 + col("Mes")).orderBy("Indice_Temporal")

# FILTRADO: ENTRENAMIENTO SOLO CON LOS ÚLTIMOS 3 MESES DE 2024
print("Filtrando ventana de entrenamiento: Últimos 3 meses de 2024...")
df_entrenamiento_3m = df_ml.filter((col("Anio") == 2024) & (col("Mes") >= 10))

# Log Transform y Features
df_entrenamiento_3m = df_entrenamiento_3m.withColumn("label_log", log(col("Total_Viajes") + 1))
ensamblador = VectorAssembler(inputCols=["Indice_Temporal"], outputCol="features")

df_train_final = ensamblador.transform(df_entrenamiento_3m)

# Entrenamiento del modelo
modelo = LinearRegression(featuresCol="features", labelCol="label_log").fit(df_train_final)

# FORECAST: ENERO, FEBRERO Y MARZO 2025
# Índices para 2025 basándonos en el inicio de 2021: Ene(49), Feb(50), Mar(51)
futuro_datos = [(49, 2025, 1, "Enero 2025"), (50, 2025, 2, "Febrero 2025"), (51, 2025, 3, "Marzo 2025")]
df_futuro = spark.createDataFrame(futuro_datos, ["Indice_Temporal", "Anio", "Mes", "Periodo"])

# Predicción
df_prediccion = modelo.transform(ensamblador.transform(df_futuro))
df_forecast = df_prediccion.withColumn("Demanda_Predicha", round(exp(col("prediction")) - 1, 0).cast("integer")) \
                           .select("Anio", "Mes", "Periodo", "Demanda_Predicha")

print("\nPREDICCIÓN (FORECAST 2025):")
df_forecast.show()

# COMPARACIÓN CON DATOS REALES (Enero y Febrero 2025)
print("Comparando con datos reales de Enero y Febrero 2025...")

# Extraemos los reales del df_ml original (donde sí hay millones de viajes)
df_reales = df_ml.filter((col("Anio") == 2025) & (col("Mes").isin(1, 2, 3))) \
                 .select(col("Anio"), col("Mes"), col("Total_Viajes").alias("Demanda_Real"))

# Cruzamos Predicción vs Realidad
df_comparativa = df_forecast.join(df_reales, ["Anio", "Mes"], "inner") \
    .withColumn("Desviacion_Abs", abs(col("Demanda_Real") - col("Demanda_Predicha"))) \
    .withColumn("Error_Pct", round((col("Desviacion_Abs") / col("Demanda_Real")) * 100, 2))

print("\nRESULTADO DE LA COMPARACIÓN (REAL vs PREDICHO):")
df_comparativa.select("Periodo", "Demanda_Real", "Demanda_Predicha", "Error_Pct").show()

ruta_processed_forecast = "abfss://nyctaxi-processed@stnycanalysis.dfs.core.windows.net/processed/df_forecast_2025/"
ruta_comparativa = "abfss://nyctaxi-processed@stnycanalysis.dfs.core.windows.net/processed/df_comparativa_real_2025/"

print("Pipeline finalizado con éxito.")

In [2]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, year, month, round, unix_timestamp, log, exp, abs
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

print("Iniciando PROCESSED: Entrenamiento de 3 KPIs (Ventana 3 meses)...")

# LECTURA Y LIMPIEZA
ruta_cuarentena_ok = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/cuarentena/datos_con_columnas/"
df_crudo = spark.read.parquet(ruta_cuarentena_ok)

df_limpio = df_crudo.dropDuplicates().dropna(subset=["PULocationID", "DOLocationID"])
df_limpio = df_limpio.withColumn("duration_min", (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60)
df_limpio = df_limpio.filter((col("duration_min") >= 0.5) & (col("duration_min") <= 300) & (col("trip_distance") > 0) & (col("total_amount") >= 2.50))

# CÁLCULO DE LOS 3 KPIs (Demanda, Ingresos y Distancia)
df_kpis = df_limpio.groupBy(year("pickup_datetime").alias("Anio"), month("pickup_datetime").alias("Mes")).agg(
    count("*").alias("Total_Viajes"),
    round(spark_sum("total_amount"), 2).alias("Ingresos_Totales"),
    round(avg("trip_distance"), 2).alias("Distancia_Media")
)

df_ml = df_kpis.withColumn("Indice_Temporal", (col("Anio") - 2021) * 12 + col("Mes")).orderBy("Indice_Temporal")

# PREPARACIÓN DE DATOS (Entrenamiento y Futuro)
print("Preparando datos de entrenamiento y futuro...")
df_entrenamiento_3m = df_ml.filter((col("Anio") == 2024) & (col("Mes") >= 10))

ensamblador = VectorAssembler(inputCols=["Indice_Temporal"], outputCol="features")
df_train_features = ensamblador.transform(df_entrenamiento_3m)

# Creamos el dataframe vacío para los próximos 3 meses
futuro_datos = [(49, 2025, 1, "Enero 2025"), (50, 2025, 2, "Febrero 2025"), (51, 2025, 3, "Marzo 2025")]
df_forecast = spark.createDataFrame(futuro_datos, ["Indice_Temporal", "Anio", "Mes", "Periodo"])
df_futuro_features = ensamblador.transform(df_forecast)

# ENTRENAMIENTO Y FORECAST BUCLE (Los 3 KPIs)
kpis_a_predecir = ["Total_Viajes", "Ingresos_Totales", "Distancia_Media"]

print("Entrenando modelos para cada KPI...")

for kpi in kpis_a_predecir:
    # Transformación Logarítmica para estabilizar el modelo
    df_train_kpi = df_train_features.withColumn("label_log", log(col(kpi) + 1))
    
    # Entrenamos el modelo específico para este KPI
    modelo = LinearRegression(featuresCol="features", labelCol="label_log").fit(df_train_kpi)
    
    # Hacemos la predicción
    df_pred = modelo.transform(df_futuro_features)
    
    # Revertimos el logaritmo y formateamos (Enteros para viajes, decimales para el resto)
    nombre_col_pred = f"Pred_{kpi}"
    if kpi == "Total_Viajes":
        df_pred = df_pred.withColumn(nombre_col_pred, round(exp(col("prediction")) - 1, 0).cast("integer"))
    else:
        df_pred = df_pred.withColumn(nombre_col_pred, round(exp(col("prediction")) - 1, 2))
        
    # Juntamos la nueva predicción a nuestra tabla final de forecast
    df_forecast = df_forecast.join(df_pred.select("Indice_Temporal", nombre_col_pred), ["Indice_Temporal"], "inner")

print("\nPREDICCIÓN A 3 MESES (FORECAST DE 3 KPIs):")
df_forecast.select("Periodo", "Pred_Total_Viajes", "Pred_Ingresos_Totales", "Pred_Distancia_Media").show()

# COMPARACIÓN CON DATOS REALES
print("Comparando Predicciones vs Realidad (Ene, Feb, Mar 2025)...")

# Extraemos los datos reales de los 3 KPIs
df_reales = df_ml.filter((col("Anio") == 2025) & (col("Mes").isin(1, 2, 3))) \
                 .select("Anio", "Mes", 
                         col("Total_Viajes").alias("Real_Total_Viajes"),
                         col("Ingresos_Totales").alias("Real_Ingresos_Totales"),
                         col("Distancia_Media").alias("Real_Distancia_Media"))

df_comparativa = df_forecast.join(df_reales, ["Anio", "Mes"], "inner")

# Calculamos el error porcentual de los 3
for kpi in kpis_a_predecir:
    col_real = f"Real_{kpi}"
    col_pred = f"Pred_{kpi}"
    col_error = f"ErrorPct_{kpi}"
    
    df_comparativa = df_comparativa.withColumn(
        col_error, round((abs(col(col_real) - col(col_pred)) / col(col_real)) * 100, 2)
    )

df_comparativa = df_comparativa.orderBy("Mes")

print("\nRESULTADO DEL ERROR DE PREDICCIÓN (%):")
df_comparativa.select("Periodo", "ErrorPct_Total_Viajes", "ErrorPct_Ingresos_Totales", "ErrorPct_Distancia_Media").show()

# GUARDADO EN CAPA PROCESSED
ruta_processed_forecast = "abfss://nyctaxi-processed@stnycanalysis.dfs.core.windows.net/processed/df_forecast_2025/"
ruta_comparativa = "abfss://nyctaxi-processed@stnycanalysis.dfs.core.windows.net/processed/df_comparativa_real_2025/"

df_forecast.write.mode("overwrite").parquet(ruta_processed_forecast)
df_comparativa.write.mode("overwrite").parquet(ruta_comparativa)

print("Pipeline finalizado con éxito. Los 3 KPIs han sido predecidos.")

In [29]:
spark.stop()